# Measurement Values (V2)

This notebook shows the kinds of values that a V2 measurement can hold, how to add limits, and how to read the values back.

- **Scalar values**: `float`, `int`, `str`, `bool`, and `Scalar` (a value with units).
- **Non-scalar values**: `Vector`, a list of numbers, `AnalogWaveform`, `ComplexWaveform`, `DigitalWaveform`, `Spectrum`, and `XYData`.
- **Limits**: an upper and lower limit stored with the measurement.

`get_measurement` and `query_measurements` include the value only for scalar values. For any value, use `read_measurement_value`.

## Prerequisites

- NI Measurement Data Services running in V2 mode
- Python environment with the `ni.datastore` package (see `../README.md`)

In [ ]:
# Perform example-specific setup. This is not needed when writing production code.
# Set MDS_V2_GRPC_TARGET (for example, "[::1]:59606") to connect to a specific V2 service.
# When it is not set, the clients use discovery to find the service.
import os

import grpc

grpc_target = os.environ.get("MDS_V2_GRPC_TARGET", "")
# Remove gRPC's 4 MB receive limit so that large values, such as long waveforms, can be read.
grpc_options = [("grpc.max_receive_message_length", -1)]
grpc_channel = grpc.insecure_channel(grpc_target, options=grpc_options) if grpc_target else None

from ni.datastore.data.v2 import DataStoreClient

data_store_client = DataStoreClient(grpc_channel=grpc_channel)

from ni.datastore.data.v2 import Step, TestResult

test_result_id = data_store_client.create_test_result(TestResult(name="Measurement Values Example"))
step_id = data_store_client.create_step(Step(name="Example Step", test_result_id=test_result_id))
print(f"Created Test Result: {test_result_id}")
print(f"Created Step: {step_id}")

## Scalar Values

A `Scalar` carries its own units, which are also saved as the measurement's `units`. For a plain value, pass `units` to `create_measurement`.

A single value is always read back as a `Scalar`, so `.value` and `.units` work for every single value.

In [ ]:
from nitypes.scalar import Scalar

scalar_values = {
    "Example Float Measurement": 0.125,
    "Example Integer Measurement": 42,
    "Example String Measurement": "nominal",
    "Example Bool Measurement": True,
    "Example Scalar Measurement": Scalar(4.98, "V"),
}

scalar_ids = {
    name: data_store_client.create_measurement(name, value, step_id)
    for name, value in scalar_values.items()
}

for name, measurement_id in scalar_ids.items():
    measurement = data_store_client.get_measurement(measurement_id)
    print(f"{name}: value={measurement.value.value!r}, units='{measurement.units}'")
    expected = scalar_values[name]
    assert measurement.value == (expected if isinstance(expected, Scalar) else Scalar(expected))

assert data_store_client.get_measurement(scalar_ids["Example Scalar Measurement"]).units == "V"

## Limits

Limits are stored with the measurement. Use `read_measurement_values` to read the value and both limits in one call. A limit that was not set is `None`.

The service stores the outcome that you supply; it does not compute it from the limits.

In [ ]:
from ni.datastore.data.v2 import Outcome

limited_measurement_id = data_store_client.create_measurement(
    "Example Limited Measurement",
    Scalar(5.01, "V"),
    step_id,
    upper_limit=5.1,
    lower_limit=4.9,
    outcome=Outcome.PASSED,
)

values = data_store_client.read_measurement_values(
    data_store_client.get_measurement(limited_measurement_id)
)
print(f"value: {values.value}")
print(f"upper limit: {values.upper_limit}")
print(f"lower limit: {values.lower_limit}")
assert values.upper_limit == Scalar(5.1) and values.lower_limit == Scalar(4.9)

value_only = data_store_client.read_measurement_values(
    data_store_client.get_measurement(limited_measurement_id), read_fields=["value"]
)
print(f"value only: {value_only.value}, upper limit: {value_only.upper_limit}")

## Non-Scalar Values

Create non-scalar values the same way. Read them back with `read_measurement_value`, optionally passing `expected_type` to check the type.

In [ ]:
from datetime import timezone

import hightime as ht
import numpy as np
from nitypes.vector import Vector
from nitypes.waveform import AnalogWaveform, ComplexWaveform, DigitalWaveform, Spectrum, Timing
from nitypes.xy_data import XYData

non_scalar_values = {
    "Example Vector Measurement": Vector([1.0, 1.25, 1.5], units="V"),
    "Example List Measurement": [1.0, 2.0, 3.0],
    "Example AnalogWaveform Measurement": AnalogWaveform(
        sample_count=3,
        raw_data=np.array([1.0, 2.0, 3.0]),
        timing=Timing.create_with_regular_interval(
            ht.timedelta(seconds=1e-3), ht.datetime.now(timezone.utc)
        ),
    ),
    "Example ComplexWaveform Measurement": ComplexWaveform.from_array_1d(
        np.array([1 + 2j, 3 + 4j]), dtype=np.complex128
    ),
    "Example DigitalWaveform Measurement": DigitalWaveform.from_lines([[0, 1], [1, 0]], signal_count=2),
    "Example Spectrum Measurement": Spectrum.from_array_1d(
        np.array([1.0, 2.0]), dtype=np.float64, start_frequency=10.0, frequency_increment=5.0
    ),
    "Example XYData Measurement": XYData.from_arrays_1d(
        [1.0, 2.0], [3.0, 4.0], np.float64, x_units="s", y_units="V"
    ),
}

non_scalar_ids = {
    name: data_store_client.create_measurement(name, value, step_id)
    for name, value in non_scalar_values.items()
}

for name, measurement_id in non_scalar_ids.items():
    measurement = data_store_client.get_measurement(measurement_id)
    read_back = data_store_client.read_measurement_value(measurement)
    print(f"{name}: {type(read_back).__name__}")
    # get_measurement does not include non-scalar values.
    assert measurement.value is None

read_back_waveform = data_store_client.read_measurement_value(
    data_store_client.get_measurement(non_scalar_ids["Example AnalogWaveform Measurement"]),
    expected_type=AnalogWaveform,
)
print(f"waveform data: {read_back_waveform.raw_data}")
assert read_back_waveform.raw_data.tolist() == [1.0, 2.0, 3.0]

# A list of numbers is read back as a Vector.
read_back_list = data_store_client.read_measurement_value(
    data_store_client.get_measurement(non_scalar_ids["Example List Measurement"]),
    expected_type=Vector,
)
print(f"list read back as: {read_back_list}")

Limits can also be non-scalar, such as a `Vector` for per-point limits:

In [ ]:
vector_limit_id = data_store_client.create_measurement(
    "Example Vector Limit Measurement",
    Vector([1.0, 2.0], units="V"),
    step_id,
    upper_limit=Vector([3.0, 3.0], units="V"),
    lower_limit=Vector([0.0, 0.0], units="V"),
)
vector_values = data_store_client.read_measurement_values(
    data_store_client.get_measurement(vector_limit_id)
)
print(f"value: {vector_values.value}")
print(f"upper limit: {vector_values.upper_limit}")
print(f"lower limit: {vector_values.lower_limit}")

## Clean up

In [ ]:
data_store_client.close()

# Perform example-specific cleanup. This is not needed when writing production code.
if grpc_channel is not None:
    grpc_channel.close()